# Evaluación de BIGRec y D3 en la GPU de Colab (frente E)

Evalúa en la GPU gratuita de Colab (T4) los modelos de BIGRec que se entrenaron en local y se subieron a Drive: **BIGRec** (`length_penalty` 1,0) y **D3** (`length_penalty` 0,0) en Video_Games y Musical_Instruments. Solo llama a funciones de `notebooks/bigrec/bigrec_baseline.py`; los hiperparámetros son los de siempre (10 beams restringidos al catálogo, lotes de 24, `max_new_tokens` 128).

- **Precisión:** la T4 no tiene bf16 en hardware, así que se evalúa en fp16. Es una desviación respecto de los autores (bf16) y queda registrada en cada `registro.json`.
- **Validación (primera sesión):** se evalúa BIGRec Toys en la T4 y se compara con la corrida local (RX 9070 XT, bf16). Si HR@10 o NDCG@10 difieren en más de `UMBRAL`, el notebook se detiene.
- **Retomar:** volver a correr todo. La evaluación se guarda por partes de 256 filas en Drive y se salta lo ya hecho.

Instrucciones completas: [`colab/README.md`](README.md).

In [ ]:
# Preparación de la sesión: correr esta celda primero en cada sesión nueva (se puede repetir sin problema)
import os
import sys
import subprocess
COMMIT = 'PENDIENTE'   # commit fijo del repo (lo deja escrito quien coordina al publicar la versión)
URL = 'https://github.com/mannicchiarico/iic3633-proyecto-latentr3'
REPO = '/content/iic3633'
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    if COMMIT == 'PENDIENTE':
        raise RuntimeError('Falta fijar COMMIT: pedirlo a quien coordina.')
    if not os.path.isdir(os.path.join(REPO, '.git')):
        subprocess.run(['git', 'clone', '-q', URL, REPO], check=True)
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '--detach', 'origin/main' if COMMIT == 'main' else COMMIT],
                   check=True)
    print('Repo en el commit', subprocess.run(['git', '-C', REPO, 'rev-parse', 'HEAD'], capture_output=True,
                                              text=True).stdout.strip())
    sys.path.insert(0, os.path.join(REPO, 'colab'))
    import preparar
    RUTAS = preparar.preparar('evaluaciones')
    RAIZ = REPO
else:
    raise RuntimeError('Este notebook es solo para Colab.')

In [ ]:
# Parámetros
DATASETS = ['Toys_and_Games', 'Video_Games', 'Musical_Instruments']   # Toys solo se usa para validar la GPU
UMBRAL = 0.003                 # diferencia máxima de HR@10 y NDCG@10 frente a la corrida local (suposición, ajustable)
CONTINUAR_SIN_VALIDAR = False  # True solo si quien coordina lo autoriza

In [ ]:
import json
import pandas as pd
import torch
from baseline_comun import precision_gpu, entorno
import bigrec_baseline as bb
from monitor import Monitor

base, IIC3633, calc_py = RUTAS['base'], RUTAS['iic3633'], RUTAS['calc_py']
SALIDA = os.path.join(IIC3633, 'checkpoints', 'bigrec')        # modelos (accesos directos) y predicciones
CARPETA_COLAB = os.path.join(IIC3633, 'colab', 'evaluaciones')  # uso de la GPU, etapas y validación
RUTA_VALIDACION = os.path.join(CARPETA_COLAB, 'validacion_t4.json')
TOYS = 'Toys_and_Games'
bb.cargar_d3(RUTAS['codigo_d3'])   # verifica el sha256 del código de D3 antes de importarlo
bb.DTYPE = precision_gpu()
print('Precisión de la evaluación:', bb.DTYPE, '(bf16 = la de los autores; fp16 = desviación registrada)')
monitor = Monitor(CARPETA_COLAB).iniciar()   # gpu_log.csv cada 60 s en Drive


def modelo_listo(salida):
    # config.json y los .safetensors legibles (se lee solo la cabecera): detecta subidas incompletas
    from safetensors import safe_open
    carpeta = os.path.join(salida, 'modelo')
    if not os.path.isfile(os.path.join(carpeta, 'config.json')):
        return False
    pesos = [f for f in os.listdir(carpeta) if f.endswith('.safetensors')]
    try:
        for archivo in pesos:
            with safe_open(os.path.join(carpeta, archivo), 'pt') as f:
                f.keys()
    except Exception:
        return False
    return bool(pesos)


def carpetas(categoria):
    return ['bigrec_t4'] if categoria == TOYS else ['bigrec', 'd3']


estado = {}
for categoria in DATASETS:
    salida = os.path.join(SALIDA, categoria)
    fila = {'modelo listo': modelo_listo(salida)}
    for carpeta in carpetas(categoria):
        pred = os.path.join(salida, 'predicciones', carpeta)
        if os.path.isfile(os.path.join(pred, 'final_result.json')):
            fila[carpeta] = 'terminada'
        elif os.path.isdir(pred):
            fila[carpeta] = f"{sum(f.startswith('parte_') for f in os.listdir(pred))} partes"
        else:
            fila[carpeta] = 'pendiente'
    estado[categoria] = fila
pd.DataFrame(estado).T

## Validación de la GPU con BIGRec Toys

Se evalúa el modelo de BIGRec Toys (el mismo que se evaluó en local) sobre todo `test` y se compara con `resultados/bigrec/Toys_and_Games/predicciones/bigrec/final_result.json`, la corrida local en bf16. Se guarda en `predicciones/bigrec_t4/`, sin tocar lo local. El resultado queda en `validacion_t4.json` y además mide el efecto de cambiar de GPU y de precisión.

In [ ]:
if TOYS in DATASETS:
    salida = os.path.join(SALIDA, TOYS)
    assert modelo_listo(salida), f'Falta el modelo de BIGRec Toys en {salida}/modelo (ver README, paso 2).'
    with monitor.etapa('Toys_and_Games bigrec_t4 (validación)'):
        ruta_t4 = bb.evaluar_categoria(TOYS, base, salida, 'bigrec', carpeta='bigrec_t4')
    info = bb.buscar_archivo(base, 'info', TOYS)
    bb.verificar_con_calc(ruta_t4, info, calc_py)   # falla si las métricas no coinciden con calc.py
    local = os.path.join(RUTAS['repo'], 'resultados', 'bigrec', TOYS, 'predicciones', 'bigrec')
    m_t4, m_local = bb.metricas(ruta_t4, info), bb.metricas(os.path.join(local, 'final_result.json'), info)
    with open(ruta_t4) as f:
        p_t4 = [fila['predict'] for fila in json.load(f)]
    with open(os.path.join(local, 'final_result.json')) as f:
        p_local = [fila['predict'] for fila in json.load(f)]
    with open(os.path.join(os.path.dirname(ruta_t4), 'registro.json')) as f:
        reg_t4 = json.load(f)
    with open(os.path.join(local, 'registro.json')) as f:
        reg_local = json.load(f)
    assert len(p_t4) == len(p_local)
    deltas = {m: m_t4[m] - m_local[m] for m in ['HR@5', 'HR@10', 'NDCG@5', 'NDCG@10']}
    validacion = {
        'ok': abs(deltas['HR@10']) <= UMBRAL and abs(deltas['NDCG@10']) <= UMBRAL, 'umbral': UMBRAL,
        'dtype_t4': reg_t4['dtype'], 'metricas_t4': m_t4, 'metricas_local': m_local, 'deltas': deltas,
        'top10_identico_%': 100 * sum(a == b for a, b in zip(p_t4, p_local)) / len(p_t4),
        'top1_identico_%': 100 * sum(a[0] == b[0] for a, b in zip(p_t4, p_local)) / len(p_t4),
        's_por_fila_t4': reg_t4['segundos'] / reg_t4['filas'],
        's_por_fila_local': reg_local['segundos'] / reg_local['filas'], 'entorno': entorno()}
    os.makedirs(CARPETA_COLAB, exist_ok=True)
    with open(RUTA_VALIDACION, 'w') as f:
        json.dump(validacion, f, indent=1, ensure_ascii=False)
    display(pd.DataFrame({'local (RX 9070 XT, bf16)': m_local, f'Colab ({reg_t4["dtype"]})': m_t4}).T[
        ['HR@5', 'HR@10', 'NDCG@5', 'NDCG@10', 'fuera del catálogo']])
    print({k: validacion[k] for k in ['ok', 'deltas', 'top10_identico_%', 'top1_identico_%', 's_por_fila_t4',
                                      's_por_fila_local']})
    if not validacion['ok'] and not CONTINUAR_SIN_VALIDAR:
        raise RuntimeError('La validación no pasó. Detenerse, correr la celda de exportar y enviar el zip a quien '
                           'coordina.')

## Evaluación de BIGRec y D3

Para cada dataset con el modelo ya en Drive: BIGRec y después D3, sobre el mismo modelo. Si el modelo aún no se subió, el dataset se salta (volver a correr más tarde).

In [ ]:
def validacion_ok():
    if not os.path.isfile(RUTA_VALIDACION):
        return False
    with open(RUTA_VALIDACION) as f:
        return json.load(f)['ok']


if not (validacion_ok() or CONTINUAR_SIN_VALIDAR):
    raise RuntimeError('Primero tiene que pasar la validación con BIGRec Toys (celda anterior).')
for categoria in [c for c in DATASETS if c != TOYS]:
    salida = os.path.join(SALIDA, categoria)
    if not modelo_listo(salida):
        print(f'{categoria}: el modelo todavía no está en {salida}/modelo; se salta.')
        continue
    for variante in ['bigrec', 'd3']:
        with monitor.etapa(f'{categoria} {variante}'):
            bb.evaluar_categoria(categoria, base, salida, variante)

## Métricas

HR@K y NDCG@K por título con el criterio de `calc.py` de los autores. Cada `final_result.json` se verifica corriendo `calc.py` sobre el mismo archivo (si da distinto, la celda falla) y los títulos fuera del catálogo deben ser 0. Se muestran junto a la Tabla 1 del paper.

In [ ]:
METRICAS = ['HR@5', 'HR@10', 'NDCG@5', 'NDCG@10']
PAPER = {('Video_Games', 'bigrec'): [0.0461, 0.0709, 0.0334, 0.0414],
         ('Video_Games', 'd3'): [0.0608, 0.0860, 0.0423, 0.0505],
         ('Musical_Instruments', 'bigrec'): [0.0938, 0.1158, 0.0807, 0.0879],
         ('Musical_Instruments', 'd3'): [0.0984, 0.1167, 0.0848, 0.0907]}
filas = []
for categoria in DATASETS:
    info = bb.buscar_archivo(base, 'info', categoria)
    for carpeta in carpetas(categoria):
        ruta = os.path.join(SALIDA, categoria, 'predicciones', carpeta, 'final_result.json')
        if not os.path.isfile(ruta):
            continue
        bb.verificar_con_calc(ruta, info, calc_py)
        m = bb.metricas(ruta, info)
        assert m['fuera del catálogo'] == 0, (categoria, carpeta, m['fuera del catálogo'])
        filas.append({'dataset': categoria, 'fuente': f'Colab: {carpeta}', **{k: m[k] for k in METRICAS}})
        if (categoria, carpeta) in PAPER:
            filas.append({'dataset': categoria, 'fuente': f'Tabla 1: {carpeta}',
                          **dict(zip(METRICAS, PAPER[(categoria, carpeta)]))})
pd.DataFrame(filas).set_index(['dataset', 'fuente']).round(4) if filas else 'Aún no hay predicciones terminadas.'

## Exportar

Arma un zip solo con los `registro.json`, los `final_result.json`, la validación y el registro de la GPU (sin pesos ni partes), lo deja en `MyDrive/IIC3633/export/` y lo descarga. Ese zip es lo que hay que enviar a quien coordina.

In [ ]:
from exportar import exportar
from google.colab import files
ruta_zip = exportar(IIC3633, 'evaluaciones')
files.download(ruta_zip)